In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

# Reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
df_lstm = pd.read_csv("PJME_hourly.csv")

df_lstm["Datetime"] = pd.to_datetime(df_lstm["Datetime"])
df_lstm = df_lstm.sort_values("Datetime").reset_index(drop=True)

df_lstm = df_lstm.dropna(subset=["PJME_MW"]).reset_index(drop=True)

values = df_lstm["PJME_MW"].values.astype(np.float32)

N = len(values)

train_end = int(N * 0.70)
val_end = int(N * 0.85)

print("Total rows:", N)
print("Training boundary:", train_end)
print("Validation boundary:", val_end)
print("Test rows:", N - val_end)

Total rows: 145366
Training boundary: 101756
Validation boundary: 123561
Test rows: 21805


In [3]:
scaler = StandardScaler()

scaler.fit(values[:train_end].reshape(-1, 1))

scaled_values = scaler.transform(
    values.reshape(-1, 1)
).flatten().astype(np.float32)

os.makedirs("lstm_checkpoints", exist_ok=True)

joblib.dump(scaler, "lstm_checkpoints/scaler.pkl")

print("Scaling completed!")

Scaling completed!


In [4]:
INPUT_LEN = 168
OUTPUT_LEN = 24

class EnergyDataset(Dataset):
    def __init__(self, data, start_indices):
        self.data = data
        self.start_indices = start_indices

    def __len__(self):
        return len(self.start_indices)

    def __getitem__(self, idx):
        start = self.start_indices[idx]

        x = self.data[start : start + INPUT_LEN]
        y = self.data[
            start + INPUT_LEN : start + INPUT_LEN + OUTPUT_LEN
        ]

        return (
            torch.tensor(x, dtype=torch.float32).unsqueeze(-1),
            torch.tensor(y, dtype=torch.float32)
        )

# Training sequences: targets entirely within training period
train_starts = np.arange(
    0, train_end - INPUT_LEN - OUTPUT_LEN + 1
)

# Validation sequences: targets entirely within validation period
val_starts = np.arange(
    train_end - INPUT_LEN,
    val_end - INPUT_LEN - OUTPUT_LEN + 1
)

# Test sequences: targets entirely within test period
test_starts = np.arange(
    val_end - INPUT_LEN,
    N - INPUT_LEN - OUTPUT_LEN + 1
)

train_dataset = EnergyDataset(scaled_values, train_starts)
val_dataset = EnergyDataset(scaled_values, val_starts)
test_dataset = EnergyDataset(scaled_values, test_starts)

print("Training sequences:", len(train_dataset))
print("Validation sequences:", len(val_dataset))
print("Testing sequences:", len(test_dataset))

Training sequences: 101565
Validation sequences: 21782
Testing sequences: 21782


In [5]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("DataLoaders ready!")

DataLoaders ready!


In [6]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64,
                 num_layers=2, output_size=24, dropout=0.2):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        last_output = out[:, -1, :]
        return self.fc(last_output)


model = LSTMModel().to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print(model)

LSTMModel(
  (lstm): LSTM(1, 64, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=64, out_features=24, bias=True)
)


In [7]:
CHECKPOINT_DIR = "lstm_checkpoints"

LAST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR, "last_checkpoint.pth"
)

BEST_MODEL_PATH = os.path.join(
    CHECKPOINT_DIR, "best_model.pth"
)

EPOCHS = 50
PATIENCE = 8

best_val_loss = float("inf")
start_epoch = 0
epochs_without_improvement = 0

# Resume if a checkpoint exists
if os.path.exists(LAST_CHECKPOINT):
    checkpoint = torch.load(
        LAST_CHECKPOINT,
        map_location=device,
        weights_only=False
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    start_epoch = checkpoint["epoch"] + 1
    best_val_loss = checkpoint["best_val_loss"]
    epochs_without_improvement = checkpoint["epochs_without_improvement"]

    print(f"Resuming from epoch {start_epoch + 1}")
else:
    print("Starting training from epoch 1")

for epoch in range(start_epoch, EPOCHS):

    # Training
    model.train()
    train_losses = []

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        predictions = model(x)
        loss = criterion(predictions, y)

        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    train_loss = np.mean(train_losses)

    # Validation
    model.eval()
    val_losses = []

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)

            predictions = model(x)
            loss = criterion(predictions, y)

            val_losses.append(loss.item())

    val_loss = np.mean(val_losses)

    # Check for improvement
    improved = val_loss < best_val_loss

    if improved:
        best_val_loss = val_loss
        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH
        )
    else:
        epochs_without_improvement += 1

    # Save latest checkpoint after every completed epoch
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_loss": best_val_loss,
        "epochs_without_improvement": epochs_without_improvement
    }, LAST_CHECKPOINT)

    # Append training log
    log_row = pd.DataFrame([{
        "Epoch": epoch + 1,
        "Train_Loss": train_loss,
        "Val_Loss": val_loss
    }])

    log_row.to_csv(
        os.path.join(CHECKPOINT_DIR, "training_log.csv"),
        mode="a",
        header=not os.path.exists(
            os.path.join(CHECKPOINT_DIR, "training_log.csv")
        ),
        index=False
    )

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Train Loss: {train_loss:.5f} | "
        f"Val Loss: {val_loss:.5f} | "
        f"Best Val: {best_val_loss:.5f}"
    )

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping triggered.")
        break

print("Training finished!")
print("Best validation loss:", best_val_loss)

Starting training from epoch 1
Epoch [1/50] Train Loss: 0.18128 | Val Loss: 0.12845 | Best Val: 0.12845
Epoch [2/50] Train Loss: 0.10690 | Val Loss: 0.11554 | Best Val: 0.11554
Epoch [3/50] Train Loss: 0.08673 | Val Loss: 0.09596 | Best Val: 0.09596
Epoch [4/50] Train Loss: 0.07838 | Val Loss: 0.09653 | Best Val: 0.09596
Epoch [5/50] Train Loss: 0.07197 | Val Loss: 0.08682 | Best Val: 0.08682
Epoch [6/50] Train Loss: 0.07984 | Val Loss: 0.09547 | Best Val: 0.08682
Epoch [7/50] Train Loss: 0.06897 | Val Loss: 0.08423 | Best Val: 0.08423
Epoch [8/50] Train Loss: 0.06625 | Val Loss: 0.08017 | Best Val: 0.08017
Epoch [9/50] Train Loss: 0.06350 | Val Loss: 0.07697 | Best Val: 0.07697
Epoch [10/50] Train Loss: 0.06205 | Val Loss: 0.08093 | Best Val: 0.07697
Epoch [11/50] Train Loss: 0.06089 | Val Loss: 0.07760 | Best Val: 0.07697
Epoch [12/50] Train Loss: 0.06122 | Val Loss: 0.08666 | Best Val: 0.07697
Epoch [13/50] Train Loss: 0.05823 | Val Loss: 0.07875 | Best Val: 0.07697
Epoch [14/50] Tr

In [8]:
model.load_state_dict(
    torch.load(BEST_MODEL_PATH, map_location=device, weights_only=True)
)

model.eval()

print("Best LSTM model loaded!")

Best LSTM model loaded!


In [9]:
all_predictions = []
all_actuals = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)

        predictions = model(x).cpu().numpy()

        all_predictions.append(predictions)
        all_actuals.append(y.numpy())

predictions_scaled = np.concatenate(all_predictions, axis=0)
actuals_scaled = np.concatenate(all_actuals, axis=0)

# Convert back to MW
predictions_mw = scaler.inverse_transform(
    predictions_scaled.reshape(-1, 1)
).reshape(-1, OUTPUT_LEN)

actuals_mw = scaler.inverse_transform(
    actuals_scaled.reshape(-1, 1)
).reshape(-1, OUTPUT_LEN)

mae = mean_absolute_error(
    actuals_mw.flatten(),
    predictions_mw.flatten()
)

rmse = np.sqrt(mean_squared_error(
    actuals_mw.flatten(),
    predictions_mw.flatten()
))

wape = (
    np.sum(np.abs(actuals_mw - predictions_mw))
    / np.sum(np.abs(actuals_mw))
) * 100

print(f"LSTM Test MAE:  {mae:.2f} MW")
print(f"LSTM Test RMSE: {rmse:.2f} MW")
print(f"LSTM Test WAPE: {wape:.2f}%")

LSTM Test MAE:  1408.33 MW
LSTM Test RMSE: 1992.24 MW
LSTM Test WAPE: 4.53%


In [10]:
pred_df = pd.DataFrame()

for i in range(OUTPUT_LEN):
    pred_df[f"Actual_Hour_{i+1}"] = actuals_mw[:, i]
    pred_df[f"Predicted_Hour_{i+1}"] = predictions_mw[:, i]

pred_df.to_csv("lstm_test_predictions.csv", index=False)

print("Predictions saved to lstm_test_predictions.csv")

Predictions saved to lstm_test_predictions.csv
